<a href="https://colab.research.google.com/github/Hakis3000/tvodo-customizer/blob/main/K%C3%B3pia_z%C3%A1pisn%C3%ADka_WaifuTanks_karty_WAI.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Waifu Tanks – karty (WAI-Illustrious)

Model **WAI-Illustrious** už pozná dievčatá na tankoch (sedí na tanku, v poklope, opretá o pás) – netreba ho nič učiť.
Waifu: sexi uniforma, veľké prsia, veľké oči, mäkký zvodný pohľad. Jedna waifu, jeden tank, jeden celistvý obrázok.

**Postup:**
1. Runtime → Change runtime type → **L4** → Save.
2. Spusti **bunku 1** (raz za reláciu, cca 5 min). Spýta sa na **Civitai API kľúč** – vlož ho (civitai.com → Settings → API Keys → Add). Ak ho nemáš, len stlač Enter – použije sa záložná kópia modelu.
3. Spusti **bunku 2** (nastavenia štýlu – nemeniť).
4. **Bunka 3:** vyber kartu a dievča v roletkách → spusti. Opakuj koľkokrát chceš.
5. **Bunka 4:** všetky karty za jednu stranu naraz. **Bunka 5:** variácie z dobrého obrázka.

Obrázky sa ukladajú na Drive do `WaifuTanks/karty/<GE|SU>/<karta>/`. Na konci: Runtime → Disconnect and delete runtime.

In [ ]:
#@title 1. Príprava (raz za reláciu)
!pip -q install -U diffusers transformers accelerate safetensors
import os, random, requests, getpass, torch, IPython.display as D
from PIL import Image
from google.colab import drive
drive.mount('/content/drive')
OUT = '/content/drive/MyDrive/WaifuTanks/karty'; os.makedirs(OUT, exist_ok=True)
from diffusers import StableDiffusionXLPipeline, StableDiffusionXLImg2ImgPipeline, EulerAncestralDiscreteScheduler

KEY = getpass.getpass('Civitai API kľúč (Enter = bez kľúča): ').strip()
pipe = None
if KEY:
    try:   # najnovšia verzia WAI-Illustrious z Civitai
        info = requests.get('https://civitai.com/api/v1/models/827184', timeout=30).json()
        ver = info['modelVersions'][0]; url = ver['downloadUrl']
        path = f"/content/wai_{ver['id']}.safetensors"
        if not os.path.exists(path):
            print('sťahujem', info['name'], ver['name'], '…')
            !wget -q -O "{path}" "{url}?token={KEY}"
        pipe = StableDiffusionXLPipeline.from_single_file(path, torch_dtype=torch.float16)
        print('model:', info['name'], ver['name'])
    except Exception as e:
        print('Civitai nevyšlo (', e, ') – použijem záložnú kópiu'); pipe = None
if pipe is None:   # kópie WAI-Illustrious na Hugging Face (bez Civitai) – skúsi novšiu, potom staršiu
    for repo in ['John6666/wai-nsfw-illustrious-v90-sdxl', 'John6666/wai-nsfw-illustrious-v70-sdxl']:
        try:
            pipe = StableDiffusionXLPipeline.from_pretrained(repo, torch_dtype=torch.float16); print('model:', repo); break
        except Exception as e: print('nedostupné:', repo)
pipe.scheduler = EulerAncestralDiscreteScheduler.from_config(pipe.scheduler.config)
pipe.to('cuda'); pipe.vae.enable_tiling()
i2i = StableDiffusionXLImg2ImgPipeline(vae=pipe.vae, text_encoder=pipe.text_encoder, text_encoder_2=pipe.text_encoder_2,
        tokenizer=pipe.tokenizer, tokenizer_2=pipe.tokenizer_2, unet=pipe.unet, scheduler=pipe.scheduler)

# ---- dlhé prompty: rozdelí tagy na kúsky po 75 tokenov (model inak číta len začiatok)
@torch.no_grad()
def _enc(text):
    tags = [t.strip() for t in text.split(',') if t.strip()]
    chunks, cur = [], []
    for t in tags:
        if cur and len(pipe.tokenizer(', '.join(cur + [t])).input_ids) - 2 > 75: chunks.append(', '.join(cur)); cur = [t]
        else: cur.append(t)
    chunks.append(', '.join(cur))
    embs, pooled = [], None
    for ch in chunks:
        a = pipe.tokenizer(ch, padding='max_length', max_length=77, truncation=True, return_tensors='pt').input_ids.to('cuda')
        b = pipe.tokenizer_2(ch, padding='max_length', max_length=77, truncation=True, return_tensors='pt').input_ids.to('cuda')
        o1 = pipe.text_encoder(a, output_hidden_states=True); o2 = pipe.text_encoder_2(b, output_hidden_states=True)
        if pooled is None: pooled = o2[0]
        embs.append(torch.cat([o1.hidden_states[-2], o2.hidden_states[-2]], -1))
    return embs, pooled
@torch.no_grad()
def embeds(pos, neg):
    pe, pp = _enc(pos); ne, npool = _enc(neg)
    empty, _ = _enc(' ')
    while len(pe) < len(ne): pe.append(empty[0])
    while len(ne) < len(pe): ne.append(empty[0])
    return dict(prompt_embeds=torch.cat(pe, 1), pooled_prompt_embeds=pp, negative_prompt_embeds=torch.cat(ne, 1), negative_pooled_prompt_embeds=npool)
print('✅ pripravené')

Mounted at /content/drive
Civitai API kľúč (Enter = bez kľúča): ··········


/usr/local/lib/python3.13/dist-packages/diffusers/utils/deprecation_utils.py:23: FutureWarning: `torch_dtype` is deprecated and will be removed in version 1.0.0. Please use `dtype` instead.
  deprecate("torch_dtype", "1.0.0", _TORCH_DTYPE_DEPRECATION_MESSAGE)


model_index.json:   0%|          | 0.00/721 [00:00<?, ?B/s]

Reconstructing (incomplete total...): |          |  0.00B /  0.00B            

Fetching 18 files:   0%|          | 0/18 [00:00<?, ?it/s]

Loading pipeline components...:   0%|          | 0/7 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/517 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/196 [00:00<?, ?it/s]

model: John6666/wai-nsfw-illustrious-v90-sdxl
✅ pripravené


In [ ]:
#@title 2. Štýl, dievčatá a karty
QUALITY = 'masterpiece, best quality, amazing quality, very aesthetic, absurdres, detailed eyes, detailed face'
# vektorový anime vzhľad – ploché farby, ostré obrysy, nie maľba
STYLE = 'flat color, anime screencap, vector art, cel shading, bold lineart, clean lineart, simple shading, vibrant colors, sharp focus, ww2'
NEG = ('bad quality, worst quality, worst detail, low quality, lowres, sketch, censor, bad anatomy, bad hands, extra digits, fewer digits, '
       'missing fingers, extra arms, extra legs, text, signature, watermark, username, multiple girls, 2girls, 3girls, '
       'painting (medium), oil painting (medium), watercolor (medium), painterly, realistic, photorealistic, 3d, textured, grainy, '
       'blurry, blurry background, depth of field, bokeh, '
       'one eye closed, wink, closed eyes, '
       'sharp eyes, narrow eyes, tsurime, glaring, angry, serious, frown, small eyes, small breasts, flat chest, fat, chubby, '
       'swastika, nazi, armband, multiple tanks, tank in background, distant tank')

# tvár: anime oči, zvodný pohľad (obe oči otvorené)
FACE = 'large eyes, anime eyes, sparkling eyes, detailed eyes, long eyelashes, both eyes open, half-closed eyes, blush, seductive smile, parted lips, looking at viewer'
# postava: veľké prsia, tehličky na bruchu, vyšportované nohy a zadok (SEXI 1–3)
BODY = {1: 'mature female, large breasts, cleavage, toned, abs, toned stomach, athletic legs, toned thighs, round ass',
        2: 'mature female, large breasts, cleavage, toned, abs, toned stomach, athletic legs, toned thighs, round ass, curvy, wide hips, seductive pose',
        3: 'mature female, huge breasts, deep cleavage, toned, defined abs, toned stomach, muscular legs, thick thighs, round ass, curvy, wide hips, seductive pose, arched back'}
# oblečenie: micro bikiny + krátky vojenský kabátik
UNIFORM = {
  'GE': 'black micro bikini, string bikini, cropped grey military jacket, open jacket, black thighhighs, garter straps, black gloves',
  'SU': 'red micro bikini, string bikini, cropped olive green military jacket, open jacket, red star, black thighhighs, garter straps',
}
GIRLS = {
  'Lili':    ('GE', 'long wavy light brown hair, brown eyes, grey garrison cap, black hair ribbons'),
  'Greta':   ('GE', 'long wavy pink hair, green eyes, black peaked cap'),
  'Hanna':   ('GE', 'short black bob hair, amber eyes, tan skin, red headband'),
  'Frieda':  ('GE', 'orange hair, twin braids, freckles, blue eyes, goggles on head'),
  'Mitzi':   ('GE', 'short silver hair, purple eyes, round eyewear, headphones'),
  'Elsa':    ('GE', 'long dark blue hair, ponytail, grey eyes, grey garrison cap'),
  'Katyuša': ('SU', 'long blonde hair, green eyes, ushanka, red star, red ribbon'),
  'Zoja':    ('SU', 'long straight black hair, light blue eyes, side cap, scarf'),
  'Nataša':  ('SU', 'brown hair, blue eyes'),
  'Daša':    ('SU', 'short red hair, green eyes, tank helmet'),
  'Mila':    ('SU', 'chestnut hair, twin braids, red bow, brown eyes, headphones'),
}
TANK = {'GE': 'tank, military vehicle, caterpillar tracks, tank turret, panzerkampfwagen iv',
        'SU': 'tank, military vehicle, caterpillar tracks, tank turret, t-34'}
# dievča je NA tanku, tank je veľký a blízko – nie len pozadie
ON = 'sitting on tank, sitting on vehicle, on vehicle, tank hull, girl on tank, tank close-up, vehicle focus'
# karta: (kľúč, dievča za Nemcov, dievča za Sovietov, scéna)
CARDS = {
  'Pohyb':    ('move',       'Frieda', 'Daša',    '{tank}, girl in tank hatch, upper body out of hatch, waving, laughing, driving, mud splash, dynamic angle, from below'),
  'Paľba':    ('fire',       'Greta',  'Zoja',    '{on}, {tank}, sitting on turret, firing, muzzle flash, smoke, covering ears, surprised'),
  'Obchvat':  ('flank',      'Lili',   'Katyuša', 'standing on tank, standing on vehicle, {tank}, pointing forward, ruined village, dust'),
  'Les':      ('woods',      'Lili',   'Katyuša', '{on}, {tank}, crossed legs, legs hanging, forest, sunlight'),
  'Bahno':    ('mud',        'Frieda', 'Daša',    '{on}, {tank}, stuck in mud, mud on body, embarrassed, pout'),
  'Pole':     ('field',      'Hanna',  'Nataša',  '{on}, {tank}, leaning back, arms behind head, wheat field, sunset, wind'),
  'Taktika':  ('tactics',    'Mitzi',  'Mila',    '{on}, {tank}, holding map, pencil, thinking, smile'),
  'Velenie':  ('leadership', 'Lili',   'Katyuša', '{tank}, girl in tank hatch, commander, upper body out of turret hatch, holding binoculars, leaning on hatch, smile, sky'),
  'Rozkaz':   ('command',    'Mitzi',  'Mila',    '{on}, {tank}, headphones, holding microphone, salute'),
  'Dym':      ('smoke',      'Greta',  'Zoja',    'standing on tank, standing on vehicle, {tank}, smoke, smoke screen, mischievous smile'),
  'Roklina':  ('gully',      'Hanna',  'Nataša',  '{on}, {tank}, ravine, rocks'),
  'Budova':   ('building',   'Elsa',   'Nataša',  '{on}, {tank}, ruined building, brick wall'),
  'Sutiny':   ('rubble',     'Elsa',   'Nataša',  '{on}, {tank}, rubble, destroyed city, dust'),
  'Rub karty':('back',       'Lili',   'Katyuša', '{on}, {tank}, looking back, over shoulder, from behind, rim light, ornate border'),
}
def build(card, side, girl=None, sexi=2, extra=''):
    key, ge, su, scene = CARDS[card]
    g = girl or (ge if side == 'GE' else su)
    gs, look = GIRLS[g]
    # tank a póza idú dopredu, aby ich model nevynechal
    p = ', '.join(x for x in ['1girl, solo', scene.format(tank=TANK[side], on=ON), look, UNIFORM[gs], BODY[sexi], FACE, extra, STYLE, QUALITY] if x)
    return key, g, p
print('štýl načítaný:', len(GIRLS), 'dievčat,', len(CARDS), 'kariet')

štýl načítaný: 11 dievčat, 14 kariet


In [ ]:
#@title 3. Jedna karta
KARTA = 'Pohyb'  #@param ['Pohyb', 'Paľba', 'Obchvat', 'Les', 'Bahno', 'Pole', 'Taktika', 'Velenie', 'Rozkaz', 'Dym', 'Roklina', 'Budova', 'Sutiny', 'Rub karty']
STRANA = 'Nemci'  #@param ['Nemci', 'Sovieti']
DIEVCA = 'Lili'  #@param ['podľa karty', 'Lili', 'Greta', 'Hanna', 'Frieda', 'Mitzi', 'Elsa', 'Katyuša', 'Zoja', 'Nataša', 'Daša', 'Mila']
VARIANTY = 3  #@param {type:"slider", min:1, max:8, step:1}
SEXI = 3  #@param {type:"slider", min:1, max:3, step:1}
NAVYSE = ''  #@param {type:"string"}
# NAVYSE = čo pridať po anglicky, napr. "night, moon" alebo "smiling, salute"

def gen(card, side, girl=None, n=4, sexi=2, extra='', seed=None):
    key, g, prompt = build(card, side, girl, sexi, extra)
    w, h = (896, 1254) if key == 'back' else (1024, 1024)
    os.makedirs(f'{OUT}/{side}/{key}', exist_ok=True)
    E = embeds(prompt, NEG); ims = []
    for i in range(n):
        s = seed if seed is not None else random.randint(0, 2**31 - 1)
        im = pipe(**E, width=w, height=h, num_inference_steps=28, guidance_scale=6.0,
                  generator=torch.Generator('cuda').manual_seed(s)).images[0]
        p = f'{OUT}/{side}/{key}/{key}_{g}_{s}.png'; im.save(p); ims.append(im); print('  uložené:', p)
    grid = Image.new('RGB', (300 * len(ims), int(300 * h / w)))
    for k, im in enumerate(ims): grid.paste(im.resize((300, int(300 * h / w))), (300 * k, 0))
    D.display(grid)

side = 'GE' if STRANA == 'Nemci' else 'SU'
gen(KARTA, side, None if DIEVCA == 'podľa karty' else DIEVCA, VARIANTY, SEXI, NAVYSE)

  0%|          | 0/28 [00:00<?, ?it/s]

There are modules in AutoencoderKL that should be kept in float32: []. Casting directly with `to()` can lead to inconsistent results; set `torch_dtype` in `from_pretrained()` instead to keep these modules in float32.
There are modules in AutoencoderKL that should be kept in float32: []. Casting directly with `to()` can lead to inconsistent results; set `torch_dtype` in `from_pretrained()` instead to keep these modules in float32.


  uložené: /content/drive/MyDrive/WaifuTanks/karty/GE/move/move_Lili_1787896090.png


  0%|          | 0/28 [00:00<?, ?it/s]

In [ ]:
#@title 4. Všetky karty za jednu stranu (cca 30–40 min)
STRANA = 'Nemci'  #@param ['Nemci', 'Sovieti']
VARIANTY = 4  #@param {type:"slider", min:1, max:8, step:1}
SEXI = 3  #@param {type:"slider", min:1, max:3, step:1}
side = 'GE' if STRANA == 'Nemci' else 'SU'
for card in CARDS:
    print('—', card); gen(card, side, None, VARIANTY, SEXI)

In [ ]:
#@title 5. Variácie z dobrého obrázka
CESTA = ''  #@param {type:"string"}
KARTA = 'Velenie'  #@param ['Pohyb', 'Paľba', 'Obchvat', 'Les', 'Bahno', 'Pole', 'Taktika', 'Velenie', 'Rozkaz', 'Dym', 'Roklina', 'Budova', 'Sutiny', 'Rub karty']
STRANA = 'Nemci'  #@param ['Nemci', 'Sovieti']
ZMENA = 0.6  #@param {type:"slider", min:0.3, max:0.85, step:0.05}
VARIANTY = 8  #@param {type:"slider", min:1, max:8, step:1}
# CESTA: vľavo priečinok 📁 → drive/MyDrive/WaifuTanks/karty/… → pravý klik na obrázok → Copy path → vlož sem
# ZMENA: 0.4 = skoro rovnaký, 0.6 = iná póza v rovnakom štýle, 0.8 = skoro nový obrázok
side = 'GE' if STRANA == 'Nemci' else 'SU'
key, g, prompt = build(KARTA, side)
src = Image.open(CESTA).convert('RGB'); E = embeds(prompt, NEG); ims = []
for i in range(VARIANTY):
    s = random.randint(0, 2**31 - 1)
    im = i2i(**E, image=src, strength=ZMENA, num_inference_steps=32, guidance_scale=6.0, generator=torch.Generator('cuda').manual_seed(s)).images[0]
    p = f'{OUT}/{side}/{key}/{key}_{g}_var{s}.png'; im.save(p); ims.append(im); print('  uložené:', p)
grid = Image.new('RGB', (300 * len(ims), 300))
for k, im in enumerate(ims): grid.paste(im.resize((300, 300)), (300 * k, 0))
D.display(grid)